# Lesson 10: Gradients and backpropagation

You have a loss (Lesson 9) that says how wrong the model is. Learning means changing the weights to make it smaller. The **gradient** tells each weight which way and how hard to move, and **backpropagation** computes it for every weight at once. This is the engine of training.

Run every cell top to bottom, then do the exercises at the end and upload this notebook for feedback and a grade.

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## A gradient is a slope

Mark a tensor with `requires_grad=True` and PyTorch will track it. After you build a loss from it and call `loss.backward()`, the gradient (the slope of the loss with respect to that value) shows up in `.grad`.

In [ ]:
w = torch.tensor([3.0], requires_grad=True)   # a weight we want the gradient for
loss = (w - 1) ** 2                            # a simple loss, smallest at w = 1

loss.backward()                                # compute d(loss)/d(w)
print('gradient:', w.grad)                     # tensor([4.]) = 2*(w-1) = 2*(3-1)
print('check 2*(w-1):', 2 * (3.0 - 1))

The gradient is 4, positive, meaning increasing `w` increases the loss, so to reduce the loss you should decrease `w` (move it toward 1).

## Step downhill, and repeat

Gradient descent nudges the weight a little in the opposite direction of its gradient, then does it again. Each pass recomputes the loss and gradient. Note `w.grad.zero_()`: PyTorch adds new gradients onto old ones, so you clear them each step.

In [ ]:
w = torch.tensor([3.0], requires_grad=True)
lr = 0.1
for step in range(6):
    loss = (w - 1) ** 2
    loss.backward()
    with torch.no_grad():        # update without tracking it in the graph
        w -= lr * w.grad
    w.grad.zero_()               # clear the gradient for the next step
    print(f'step {step}: w={w.item():.3f}, loss={loss.item():.4f}')

`w` marches toward 1 and the loss falls toward 0. That is training in miniature.

## Backprop fills a gradient for every weight

A real model has many weights. `loss.backward()` uses the chain rule to fill in a gradient for each one in a single backward pass. You never compute these by hand.

In [ ]:
model = nn.Linear(4, 2)
x = torch.randn(1, 4)
target = torch.tensor([1])

loss = F.cross_entropy(model(x), target)
loss.backward()
print('weight grad shape:', model.weight.grad.shape)   # (2, 4): one per weight
print('bias grad shape:  ', model.bias.grad.shape)     # (2,)

## The optimizer takes the step for you

`torch.optim` wraps the update so you do not touch each weight by hand. Hand it the parameters and a learning rate, then the training rhythm is three calls per step: `zero_grad()`, `backward()`, `step()`. Here we overfit a single example and watch the loss drop.

In [ ]:
model = nn.Linear(4, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)   # SGD = plain gradient descent
x = torch.randn(1, 4)
target = torch.tensor([1])

for step in range(6):
    optimizer.zero_grad()                       # 1. clear old gradients
    loss = F.cross_entropy(model(x), target)
    loss.backward()                             # 2. backprop fills every .grad
    optimizer.step()                            # 3. nudge every weight downhill
    print(f'step {step}: loss={loss.item():.4f}')

SGD is plain gradient descent. **Adam** and **AdamW** are smarter variants that adapt the step per weight and usually train faster; they are drop-in (same three calls). You will use Adam for MNIST and AdamW for the GPT. Wrap these three calls in a loop over your data and you have training, which is exactly the next lesson.

## Exercises

Do these in the cells below, then upload the notebook. Only your code under each `# YOUR CODE HERE` line is graded.

In [ ]:
# E1: Autograd vs the hand derivative. Make w = torch.tensor([5.0], requires_grad=True), set
#     loss = (w - 2)**2, call loss.backward(), and print w.grad. Then compute the derivative
#     yourself, 2*(w - 2), and confirm the two agree.
# YOUR CODE HERE (write your answer below)


In [ ]:
# E2: One gradient-descent step, and check it helped. Using w and w.grad from E1, step
#     w -= 0.1 * w.grad inside a torch.no_grad() block. Print the new w, and print the loss
#     (w - 2)**2 BEFORE and AFTER the step to confirm it went down.
# YOUR CODE HERE (write your answer below)


In [ ]:
# E3: Let an optimizer take the step. Create model = nn.Linear(3, 2) and
#     optimizer = torch.optim.SGD(model.parameters(), lr=0.5). With x = torch.randn(1, 3) and
#     target = torch.tensor([0]), run one training step (zero_grad, loss = F.cross_entropy(
#     model(x), target), backward, step) and print the loss. Then print model.weight.grad to
#     confirm backprop filled in a gradient for every weight.
# YOUR CODE HERE (write your answer below)


In [ ]:
# E4: Overfit one example. Repeat the E3 step in a loop of 20 iterations on the SAME x and
#     target, printing the loss each time so you see it fall. After the loop, print the model's
#     predicted class for x with model(x).argmax(dim=-1) and check it now matches the target.
# YOUR CODE HERE (write your answer below)
